In [1]:
# Cell 1: Setup and Configuration
import sys
from pathlib import Path
import pandas as pd
import itertools
import time

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.gee_sampler import CMIP6Sampler

# --- ENSEMBLE CONFIGURATION ---
# Toggle True/False to control what gets extracted
RUN_HISTORICAL = True   # 1985-2014 (For ML Bias Correction Validation)
RUN_NEAR_FUTURE = True  # 2025-2050 (Actionable Planning)
RUN_FAR_FUTURE = True   # 2070-2099 (Long-term Risk)

# VERIFIED GEE MODEL STRINGS for NASA/GDDP-CMIP6
MODELS = [
    "ACCESS-CM2", 
    "MPI-ESM1-2-HR", 
    "MRI-ESM2-0"
]

# VERIFIED GEE SCENARIO STRINGS for NASA/GDDP-CMIP6
SCENARIOS = [
    "historical", 
    "ssp245", 
    "ssp585"
]

TIMEFRAMES = {}
if RUN_HISTORICAL: TIMEFRAMES["historical"] = ("1985-01-01", "2014-12-31")
if RUN_NEAR_FUTURE: TIMEFRAMES["near_future"] = ("2025-01-01", "2050-12-31")
if RUN_FAR_FUTURE: TIMEFRAMES["far_future"] = ("2070-01-01", "2099-12-31")

# Paths
GAUGE_CSV = PROJECT_ROOT / "data" / "gauges" / "sarawak_all_gauges.csv" # UPDATE THIS PATH
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "processed" / "cmip6_ensemble"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"🚀 Ready to extract {len(MODELS) * len(SCENARIOS) * len(TIMEFRAMES)} combinations.")
print(f"   Models: {MODELS}")
print(f"   Scenarios: {SCENARIOS}")
print(f"   Timeframes: {list(TIMEFRAMES.keys())}")

c:\Users\User\miniconda3\envs\hydro-ml\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.17) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


🚀 Ready to extract 27 combinations.
   Models: ['ACCESS-CM2', 'MPI-ESM1-2-HR', 'MRI-ESM2-0']
   Scenarios: ['historical', 'ssp245', 'ssp585']
   Timeframes: ['historical', 'near_future', 'far_future']


In [2]:
# Cell 2: The Extraction Loop
results = []

for model, scenario, (tf_name, (start, end)) in itertools.product(MODELS, SCENARIOS, TIMEFRAMES.items()):
    filename = f"sarawak_cmip6_{model}_{scenario}_{tf_name}.parquet"
    output_path = OUTPUT_DIR / filename
    
    print(f"\n{'='*60}")
    print(f"STARTING: {model} | {scenario} | {tf_name} ({start} to {end})")
    print(f"{'='*60}")
    
    try:
        sampler = CMIP6Sampler(
            gauge_csv=str(GAUGE_CSV),
            col_station="station_id", # UPDATE if your CSV column is different
            col_lat="latitude",            # UPDATE if your CSV column is different
            col_lon="longitude",            # UPDATE if your CSV column is different
            start_date=start,
            end_date=end,
            project_id="ee-azaleakamellia", # UPDATE with your GEE Project ID
            scenario=scenario,
            model=model
        )
        
        df = sampler.extract_future_climate(str(output_path))
        results.append(output_path)
        print(f"✅ COMPLETED: {filename}")
        
    except Exception as e:
        print(f"❌ FAILED: {model} | {scenario} | {tf_name}")
        print(f"Error: {e}")
        
    # Small pause to let GEE breathe between heavy extractions
    time.sleep(2) 

print(f"\n🎉 EXTRACTION COMPLETE. Successfully saved {len(results)} files.")


STARTING: ACCESS-CM2 | historical | historical (1985-01-01 to 2014-12-31)
✅ GEE initialized: ee-azaleakamellia
✅ CMIP6 Sampler initialized: Scenario='historical' | Model='ACCESS-CM2'

🚀 CMIP6 Extraction | Scenario: historical | Model: ACCESS-CM2
------------------------------------------------------------
  -> Processing year 1985...
  -> Processing year 1986...
  -> Processing year 1987...
  -> Processing year 1988...
  -> Processing year 1989...
  -> Processing year 1990...
  -> Processing year 1991...
  -> Processing year 1992...
  -> Processing year 1993...
  -> Processing year 1994...
  -> Processing year 1995...
  -> Processing year 1996...
  -> Processing year 1997...
  -> Processing year 1998...
  -> Processing year 1999...
  -> Processing year 2000...
  -> Processing year 2001...
  -> Processing year 2002...
  -> Processing year 2003...
  -> Processing year 2004...
  -> Processing year 2005...
  -> Processing year 2006...
  -> Processing year 2007...
  -> Processing year 2008

In [ ]:
# Cell 3: Combine into Master Dataset
master_files = list(OUTPUT_DIR.glob("sarawak_cmip6_*.parquet"))

if master_files:
    print(f"Combining {len(master_files)} files into master dataset...")
    df_list = [pd.read_parquet(f) for f in master_files]
    master_df = pd.concat(df_list, ignore_index=True)
    
    master_path = OUTPUT_DIR / "sarawak_cmip6_master_ensemble.parquet"
    master_df.to_parquet(master_path, compression="snappy", index=False)
    
    print(f"✅ Master Ensemble Saved: {master_path}")
    print(f"   Total Rows: {len(master_df)}")
    print(f"   Models: {master_df['model'].unique()}")
    print(f"   Scenarios: {master_df['scenario'].unique()}")
else:
    print("No files found to combine.")